In [1]:
import torch
import selfies as sf
from rdkit import Chem
from model_architecture_yes_chemberta import Transformer,predict 
from rdkit.Chem import Draw
from rdkit import Chem
from rdkit.Chem import Crippen, QED
from rdkit.Chem import Descriptors
from rdkit.Chem import Lipinski
from transformers import AutoTokenizer
from rdkit.Chem.Scaffolds import MurckoScaffold

d:\minimax\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\minimax\.venv\lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
# cuda 있으면 cuda 쓰고 없으면 cpu 쓰기

model_name = "seyonec/ChemBERTa-zinc-base-v1"
chem_tokenizer = AutoTokenizer.from_pretrained(model_name)

# 체크포인트 로드
checkpoint = torch.load('D:\minimax\molecule_generation\\transformer_parameter\model_checkpoint_yes_chemberta.pt', map_location=device, weights_only=False)
token2id = checkpoint['token2id']

config = checkpoint['config'].copy()
config['max_len'] = checkpoint['max_len']

# 모델 생성
model = Transformer(**config) # 모델 파라미터 정보

# 학습된 파라미터 로드
model.load_state_dict(checkpoint['model_state_dict'], strict=False)

d:\minimax\.venv\lib\site-packages\torch\nn\modules\transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


<All keys matched successfully>

In [3]:
import pubchempy as pcp
from rdkit.Chem import inchi

def find_molecule_exists(mol): # 새로 만들어진 물질이 기존에 존재하는 것인지 아닌지 확인
	inchikey = inchi.MolToInchiKey(mol)
	results = pcp.get_compounds(inchikey, "inchikey") # inchikey로 검색해서 결과가 존재하면 기존에 분자가 이미 있는 것
	if not results:
		return None

#### 리핀스키 5규칙 
- 우리가 먹는 약에는 공통적인 특성이 존재
	- (1)분자량은 500 달톤 이하 : Descriptors.MolWt(mol)
	- (2)logP <5 : Crippen.MolLogP(mol)
	- (3)수소 결합 주개가 5개 이하 
	- (4)수소 결합 받개가 10개 이하

- 이건 걍 내가 넣고 싶은 거
	- QED 계산 : 화합물이 약처럼 될 가능성을 수치화(0~1까지)
	- qed = QED.qed(mol)

In [4]:
def isit_available_medicine(mol):
	# logp 계산 : 분자가 수용성인지, 지용성인지
	logp = Crippen.MolLogP(mol)

	# QED 계산 : 화합물이 약처럼 될 가능성을 수치화(0~1까지)
	qed = QED.qed(mol)

	molecule_weight = Descriptors.MolWt(mol) # 500 이하

	hbd = Lipinski.NumHDonors(mol) # 수소 결합 주개(5개 이하)
	hba = Lipinski.NumHAcceptors(mol) # 수소 결합 받개(10개 이하)
	
	return [molecule_weight, logp, qed, hbd, hba]

In [5]:
def extract_scaffold(smiles):
    mol = Chem.MolFromSmiles(smiles)
    scaffold = MurckoScaffold.GetScaffoldForMol(mol)
    cano_scaffold = Chem.MolToSmiles(scaffold, canonical=True)
    return cano_scaffold

In [9]:
def is_chemically_valid(smiles): # 분자의 화학적 유효성 검사
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return False
    try:
        Chem.SanitizeMol(mol)
        Chem.MolToSmiles(mol, canonical=True)
    except:
        return False
    return True

In [10]:
lst = []

smiles = 'Cn1cnc2c1c(=O)n(c(=O)n2C)C' # caffeine
cano_scaffold = extract_scaffold(smiles)

scaffold_token = chem_tokenizer(
        cano_scaffold,
        padding="max_length",
        truncation=True,
        max_length=128,
        add_special_tokens=True,
        return_tensors="pt"
    )["input_ids"]

scaffold_token = torch.tensor(scaffold_token, dtype=torch.long, device=device)

for i in range(1000):
	pred = predict(model, scaffold_token) 
	decoded = chem_tokenizer.decode(pred[0],skip_special_tokens=True)
	mol = Chem.MolFromSmiles(decoded) 
	mol_img = Draw.MolToImage(mol) # 분자 이미지
	if is_chemically_valid(decoded) and find_molecule_exists(mol) is None: # 만약 분자가 기존에 없는 것이라면
		medicine_standard = isit_available_medicine(mol) # 약이 될 수 있는지 관련 지표를 구해서 
		lst.append([f'new molecule{i}',decoded] + medicine_standard) # DB에 저장

C:\Users\amysm\AppData\Local\Temp\ipykernel_14812\2269964460.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  scaffold_token = torch.tensor(scaffold_token, dtype=torch.long, device=device)


In [11]:
import pandas as pd 

molecule_generation = pd.DataFrame(lst,columns=['molecule_name','new_smiles','molecule_weight', 'logP', 'QED', 'hbd', 'hba'])

In [12]:
molecule_generation.head()

,molecule_name,new_smiles,molecule_weight,logP,QED,hbd,hba
0,new molecule0,CC(C)n1c(=O)c2c(n(C)c(=O)n(C)c(=O)n2C)n(C)c1=O,309.326,0.1857,0.740405,0,6
1,new molecule1,CC(C)n1c(=O)c2c(n(C)c(=O)n(C)c(=O)n2C)n(C)c1=O,309.326,0.1857,0.740405,0,6
2,new molecule2,CC(C)n1c(=O)c2c(n(C)c(=O)n(C)c(=O)n2C)n(C)c1=O,309.326,0.1857,0.740405,0,6
3,new molecule3,CC(C)n1c(=O)c2c(n(C)c(=O)n(C)c(=O)n2C)n(C)c1=O,309.326,0.1857,0.740405,0,6
4,new molecule4,CC(C)n1c(=O)c2c(n(C)c(=O)n(C)c(=O)n2C)n(C)c1=O,309.326,0.1857,0.740405,0,6


In [13]:
molecule_generation.to_csv('transformer_predict_result_yes_chemberta.csv',index=False)